# Unified Cross-Domain Mathematics

**The fuselk thesis:** no existing open toolchain (OpenMC, IMAS, BLUEMIRA, BOUT++, SOLPS) combines **live plasma diagnostics**, **MHD precursor geometry**, **hierarchical RL control**, **phase-separated edge PDEs**, and **muon-catalyzed fuel-cycle kinetics** in one mathematical object.

This notebook is the **master theory document** — every governing equation, proof sketch, and coupling term from [VISION.md](../VISION.md), wired to `src/deepiri_fuselk/`.

## Abstract state-space

Define the unified fuselk state on a tokamak pulse:

$$\mathcal{U}(t) = \bigl(\underbrace{\mathbf{x}_{Kalman}}_{\text{HELIX}},\; \underbrace{\mathbf{h}_{focal}}_{\text{HQRM+attention}},\; \underbrace{\mathbf{u}_{PDE}}_{\text{oil–water}},\; \underbrace{\mathbf{N}_\mu}_{\text{muon ODE}},\; \underbrace{\mathbf{a}_{ctrl}}_{\text{Venturi}}\bigr)$$

The **closed-loop evolution** is a composition of operators (not a single PDE):

$$\mathcal{U}_{k+1} = \Phi_{ctrl}\!\circ\, \Phi_{pred}\!\circ\, \Phi_{HELIX}\!\circ\, \mathcal{D}_k$$

where $\mathcal{D}_k$ is the diagnostic injection (ECE/SXR) and $\Phi_{pred}$ fuses ELM + MHD + HELIX into $P_{dis}$.

**Novelty claim (precise):** fuselk is the first open repository where $\Phi_{ctrl}$ is constrained by both $P_{dis}$ *and* fuel-cycle feasibility $(\mathrm{TBR}, \mathrm{Pe}, N_{fus,\mu})$ — coupling control theory to nuclear fuel engineering.


In [ ]:
import sys
from pathlib import Path

repo = Path.cwd()
if not (repo / "src" / "deepiri_fuselk").exists() and (repo.parent / "src" / "deepiri_fuselk").exists():
    repo = repo.parent

try:
    import deepiri_fuselk  # noqa: F401
except ImportError:
    sys.path.insert(0, str(repo / "src"))
    import deepiri_fuselk  # noqa: F401

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")
%config InlineBackend.figure_formats = ["retina"]

from IPython.display import display
import sympy as sp
sp.init_printing()

## I. Oil–water barrier (plasma edge + fuel cycle)

Six coupled fields on $x \in [0,L]$ — see notebook `01` for numerics.

**Key scalings:**
$$\delta = \left(\frac{D_p D_v}{\alpha^2 n_0 n_w}\right)^{1/4}, \qquad \mathrm{Pe} = \frac{v_v \delta}{D_T}$$

**Theorem (extraction dominance, sketch).** If $\mathrm{Pe} > 1$ and $S_T(x) \ge 0$ with wall sink, the Peclet length $\ell_P = D_T/v_v < \delta$ implies tritium boundary layer attaches to the wall branch. Wall flux is advection-dominated ⇒ **outward fuel extraction** (not implemented in neutronics-only codes).


In [ ]:
from deepiri_fuselk.physics import PDEParameters, interface_thickness, peclet_criterion, solve_oil_water_steady

x, D_T, v_v, delta = sp.symbols("x D_T v_v delta", positive=True)
Pe = v_v * delta / D_T
display(sp.Eq(sp.Symbol("Pe"), Pe))

params = PDEParameters()
print(f"δ = {interface_thickness(params):.4f}, Pe = {peclet_criterion(params):.3f}")
steady = solve_oil_water_steady(n_grid=64)


## II. HELIX + HQRM (diagnostic differential geometry)

**Kalman state** $\mathbf{x}=[\theta,\omega,A,\phi]^\top$ with $F(\Delta t)$ as in notebook `02`.

**Boozer unwrap:** $\phi = \theta / q(r)$.

**HQRM lock:** $\mathrm{Var}(h_i) < 0.07$ over 49 field-aligned squares.

**Lock-in as matched filter:** coherent component at $\omega$ extracted by
$$\hat{s}_{coh} = \frac{1}{N}\sum_i s_i e^{-i\theta_i} \cdot e^{i\hat{\theta}}$$
Turbulence is incoherent across rotations ⇒ variance reduction $\propto 1/\sqrt{N_{cycles}}$.


In [ ]:
from deepiri_fuselk.helix import HelixEngine, boozer_map, field_line_pitch, q_profile
from deepiri_fuselk.sim import generate_ece_shot

shot = generate_ece_shot(40, seed=3)
helix = HelixEngine().process(shot.heat_field, shot.raw_signal, shot.angles)

r = np.linspace(0.2, 1.0, 50)
theta_b, phi_b = boozer_map(r, np.zeros_like(r))
print(f"HELIX SNR={helix.phase_locked_snr:.2f}, HQRM var={helix.hqrm.heat_variance:.4f}, converged={helix.hqrm.converged}")


## III. Muon master equation + stripping trifecta

**Population ODEs** (extended master equation with external field coupling):

$$\dot{\mathbf{N}} = A(R_{strip})\, \mathbf{N} + \mathbf{b}$$

where $R_{strip} = R_{col} + R_{photon} + R_{proton} + R_{cyclotron}$ and $\omega_{eff} = \omega_0(1-R_{strip})$.

**Breakeven:** $N_{fus,\mu} \ge E_\mu / E_{fus} \times (1/\omega_{eff}) \approx 284$.

**Cross-domain coupling:** muon source $S_\mu(x)$ enters tritium PDE; tritium breeding feeds $\mathrm{TBR}$ in $\mathcal{F}$.


In [ ]:
from deepiri_fuselk.muon import BREAKEVEN_FUSIONS, RateNetworkParams, run_rate_network
from deepiri_fuselk.muon.stripping_orchestrator import run_stripping_trifecta

E_mu, E_fus = 4.1e9, 17.6e6  # eV
naive_be = E_mu / E_fus
print(f"Energy-ratio breakeven (naive): {naive_be:.0f}")
print(f"Operational threshold in code: {BREAKEVEN_FUSIONS:.0f}")

configs = [
    ("baseline", RateNetworkParams(R_photon=0, R_proton=0)),
    ("photon+proton", RateNetworkParams(R_photon=0.5, R_proton=0.3)),
]
for name, p in configs:
    r = run_rate_network(params=p)
    print(f"{name:14s} N_fus,μ={r.fusions_per_muon:6.1f}  breakeven={r.breakeven}")

tri = run_stripping_trifecta()
print(f"Trifecta projected: {tri.projected_fpm:.1f} (margin {tri.margin_to_breakeven:+.1f})")


## IV. Venturi hierarchical control + disruption fusion

**Nested policy:**
$$\mathbf{a}^* = \arg\max_{\mathbf{a} \in \mathcal{A}(prior)} \; \mathcal{R}(Q, \mathbf{a}) \quad \text{s.t. watchdog}$$

**Disruption functional** (implemented in `disruption_detector.py`):
$$P_{dis} = \mathrm{clip}(0.45 P_{ELM} + 0.35 P_{MHD} + 0.2 P_{HELIX}, 0, 1)$$

**Traffic congestion:** $\chi = Q_{peak}/Q_{eng}$ — SOL modeled as a **router**, not a passive boundary.

This couples **stochastic control** (RL), **Bayesian inference** (slow prior), and **MHD stability theory** ($q_{min}$, $\beta_N$) — three domains no competitor unifies.


In [ ]:
from deepiri_fuselk.control.venturi_controller import VenturiController
from deepiri_fuselk.models.disruption_detector import DisruptionDetector
from deepiri_fuselk.models.elm_predictor import ELMPredictor
from deepiri_fuselk.sim.fusion_kpis import mhd_stability_margin

venturi = VenturiController()
detector = DisruptionDetector(ELMPredictor())

# MHD margin alone
for q_min, beta in [(2.8, 2.0), (1.5, 3.5)]:
    risk = mhd_stability_margin(q_min, beta)
    print(f"q_min={q_min}, βN={beta} → MHD risk={risk:.2f}")

assess = detector.assess(helix, q_min=1.9, beta_n=3.1)
print(f"Fused P_dis={assess.probability:.2f} → {assess.recommended_action}")


## V. Master coupling — composite fusion functional

$$\boxed{\mathcal{F} = \sum_i w_i \, \phi_i(\mathcal{U})}$$

| Term $\phi_i$ | Weight | Domain |
|---------------|--------|--------|
| $f_{TBR}$ | 0.25 | Nuclear engineering |
| $f_{ELM}$ | 0.20 | Plasma MHD + ML |
| $f_{div}$ | 0.15 | Heat exhaust geometry |
| $1 - P_{dis}$ | 0.20 | Control + stability |
| $f_\mu$ | 0.10 | Particle physics |
| $f_{SNR}$ | 0.10 | Signal processing |

**Well-posedness sketch:** Each $\Phi$ is Lipschitz on bounded diagnostic states; watchdog enforces compact action set; hence $\mathcal{U}_k$ remains bounded for finite horizon — sufficient for digital-twin stepping (not a Cauchy blow-up PDE, but a **hybrid dynamical system**).

### Comparison to status quo

| Framework | Diagnostics | Control | Fuel cycle | Muon catalysis |
|-----------|-------------|---------|------------|----------------|
| OpenMC | — | — | neutronics only | — |
| IMAS | schema | — | — | — |
| BLUEMIRA | — | — | steady design | — |
| **fuselk** | HELIX+HQRM | Venturi RL | oil–water PDE | rate network + trifecta |


In [ ]:
from deepiri_fuselk.physics import PDEParameters, interface_thickness, peclet_criterion, solve_oil_water_steady
from deepiri_fuselk.muon import RateNetworkParams, run_rate_network
from deepiri_fuselk.sim.fusion_cell import FusionCell

cell = FusionCell(grid_size=20, train_elm=False)
_, report = cell.run(n_steps=30, seed=99)

print("=== Unified fuselk report ===")
print(f"F = {report.fusion_score:.3f}")
print(f"  TBR={report.fuel_cycle.tritium_breeding_ratio:.3f}, Pe={report.fuel_cycle.peclet_number:.2f}")
print(f"  μ: {report.muon_cycle.fusions_per_muon:.1f} fusions/μ (breakeven={report.muon_cycle.breakeven})")
print(f"  ELM-free={report.elm_free_fraction:.2f}, P_dis={report.disruption_risk:.2f}")

# Sensitivity: Pe vs muon strip rate (abstract cross-domain plane)
pe_grid = np.linspace(0.5, 2.0, 8)
strip_grid = np.linspace(0, 0.6, 8)
Z = np.zeros((len(strip_grid), len(pe_grid)))
for i, rs in enumerate(strip_grid):
    for j, pe_target in enumerate(pe_grid):
        v_v = pe_target * 0.02 / 0.05  # rough scale to hit Pe
        p = PDEParameters(v_v=v_v)
        tbr = 1.0  # placeholder scale
        mu = run_rate_network(params=RateNetworkParams(R_photon=rs, R_proton=0.2)).fusions_per_muon / 284
        Z[i, j] = 0.5 * min(1, tbr) + 0.5 * min(1, mu)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(Z, origin="lower", aspect="auto", cmap="viridis",
               extent=[pe_grid[0], pe_grid[-1], strip_grid[0], strip_grid[-1]])
ax.set_xlabel("Peclet number (fuel extraction)"); ax.set_ylabel("Photon strip rate")
ax.set_title("Cross-domain feasibility plane (schematic)")
plt.colorbar(im, label="combined fuel-cycle score")
plt.tight_layout()
plt.show()


## VI. Novel contributions checklist (VISION § Novel Mathematical Contributions)

| # | Contribution | Equation object | In repo |
|---|-------------|-----------------|---------|
| 1 | Coupled oil–water PDE | 6-field system + $\delta$, Pe | `physics/`, `barrier/` |
| 2 | Peclet extraction | Pe $> 1$ criterion | `peclet_criterion()` |
| 3 | HQRM geometry | shear-split + pitch rotation | `helix/helical_quadtree.py` |
| 4 | 7×7 variance lock | Var $< 0.07$ | `run_hqrm()` |
| 5 | Spiral attention | helical RoPE bias | `focal/spiral_attention.py` |
| 6 | Kalman phase lock | EKF on $[\theta,\omega,A,\phi]$ | `helix/kalman_tracker.py` |
| 7 | Muon rate network | 5-pop ODE + $R_{strip}$ | `muon/rate_network.py` |
| 8 | Hierarchical RL + traffic | $\mathcal{R}$, $\chi$ | `control/venturi_controller.py` |
| 9 | Disruption fusion | $P_{dis}$ functional | `models/disruption_detector.py` |
| 10 | Composite $\mathcal{F}$ | weighted KPI score | `sim/fusion_kpis.py` |

**What remains research-grade (not proven here):** formal existence/uniqueness of the full 6-field PDE, JAX GPU HQRM $<1$ms claim, experimental validation of muon trifecta rates. The notebooks implement **simulation-grade** mathematics with explicit proof sketches where rigorous closure is claimed.
